## Tahap 1 – Penyimpanan Terdistribusi dan Ingesti (HDFS)

* Buat struktur direktori di HDFS, misalnya /tidarmart/raw/transaksi, /tidarmart/raw/pelanggan,
/tidarmart/raw/produk, /tidarmart/stream_input, /tidarmart/curated, /tidarmart/models.
* Unggah seluruh data ke HDFS. Tampilkan bukti ukuran, jumlah blok, dan faktor replikasi (hdfs dfs -du -
h, hdfs fsck).
* Baca seluruh data dari HDFS (hdfs://localhost:9000/...) bukan dari path lokal pada tahap-tahap berikutnya.
* Jelaskan: mengapa data mentah dan data bersih dipisah direktori, dan apa arti replikasi = 1 pada VM
pseudo-distributed dibanding klaster nyata.

In [7]:
!ls data_proyek


pelanggan.json	produk.json  stream_input  transaksi


In [14]:
# Membuat folder hdfs
!hdfs dfs -mkdir -p /tidarmart/raw/transaksi
!hdfs dfs -mkdir -p /tidarmart/raw/pelanggan
!hdfs dfs -mkdir -p /tidarmart/raw/produk
!hdfs dfs -mkdir -p /tidarmart/stream_input
!hdfs dfs -mkdir -p /tidarmart/curated
!hdfs dfs -mkdir -p /tidarmart/models

# Mengunggah
!hdfs dfs -put ~/praktikum-bigdata/proyek-akhir/data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
!hdfs dfs -put ~/praktikum-bigdata/proyek-akhir/data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
!hdfs dfs -put ~/praktikum-bigdata/proyek-akhir/data_proyek/produk.json /tidarmart/raw/produk/

# Cek ukuran, jumlah blok, dan faktor replikasi (wajib dilampirkan buktinya)
!hdfs dfs -du -h /tidarmart/raw
!hdfs fsck /tidarmart/raw -files -blocks

put: `/tidarmart/raw/transaksi/transaksi_part1.csv': File exists
put: `/tidarmart/raw/transaksi/transaksi_part2.csv': File exists
put: `/tidarmart/raw/transaksi/transaksi_part3.csv': File exists
put: `/tidarmart/raw/transaksi/transaksi_part4.csv': File exists
put: `/tidarmart/raw/pelanggan/pelanggan.json': File exists
put: `/tidarmart/raw/produk/produk.json': File exists
6.0 M    6.0 M    /tidarmart/raw/pelanggan
684.6 K  684.6 K  /tidarmart/raw/produk
187.2 M  187.2 M  /tidarmart/raw/transaksi
Connecting to namenode via http://localhost:9870/fsck?ugi=haris&files=1&blocks=1&path=%2Ftidarmart%2Fraw
FSCK started by haris (auth:SIMPLE) from /127.0.0.1 for path /tidarmart/raw at Thu Nov 26 23:07:02 WIB 2026

/tidarmart/raw <dir>
/tidarmart/raw/pelanggan <dir>
/tidarmart/raw/pelanggan/pelanggan.json 6308124 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-2097439404-127.0.1.1-1789215465808:blk_1073741924_1100 len=6308124 Live_repl=1

/tidarmart/raw/produk <dir>
/tidarmart/raw/produk/

## Tahap 2: Profiling Kualitas Data (Veracity)
* Hitung jumlah baris, jumlah order_id unik, persentase nilai kosong per kolom, jumlah harga negatif,
distribusi qty (temukan outlier dengan kuantil, jenis nya bebas), dan variasi penulisan kota.
* Sajikan hasil profiling dalam satu tabel ringkasan “sebelum pembersihan”.
* Untuk profil pelanggan dan produk: tunjukkan bahwa atribut produk bervariasi menurut kategori (berapa
persen produk memiliki setiap atribut) dan bagaimana field tags diperlakukan.

## Tahap 3 – ETL dan Penyimpanan Kolumnar (Parquet)
* Bersihkan transaksi sesuai temuan Tahap 2: deduplikasi, penanganan nilai kosong, harga negatif, outlier
qty, normalisasi nama kota. Setiap aturan harus disertai alasan (dibuang, diisi, atau dikoreksi, dan
mengapa).
* Tambahkan kolom turunan: total_nilai (qty × harga_satuan × (1 − diskon/100)), bulan,
hari_dalam_minggu, flag_batal.
* Simpan hasil ke HDFS /tidarmart/curated dalam format Parquet dengan partitionBy yang Anda pilih dan
justifikasikan (misalnya kota atau bulan).
• Bandingkan CSV dan Parquet: ukuran di HDFS dan waktu eksekusi satu query agregasi yang sama.
Sajikan dalam tabel.
• Tampilkan laporan “sesudah pembersihan” berdampingan dengan laporan sebelumnya.